# Auto-pipeline CSV: Limpieza, Codificación, Modelos y Gráficos

Este notebook implementa un flujo completo:

1. Cargar un archivo CSV.
2. Limpiar los datos (NaN, duplicados).
3. Transformar variables numéricas (imputación + normalización).
4. Transformar variables categóricas (one-hot encoding).
5. Detectar si el problema es de clasificación o regresión.
6. Probar varios modelos, compararlos y elegir el mejor.
7. Generar gráficos exploratorios y de desempeño del modelo.
8. Guardar un CSV limpio/codificado/normalizado.

Úsalo como guía de práctica de los módulos 3 a 7.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Modelos de regresión
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

# Métricas
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score

%matplotlib inline
sns.set()


In [2]:
def limpiar_y_normalizar(df):
    """
    Paso a paso:
    1) Eliminar columnas/filas completamente vacías y filas duplicadas.
    2) Separar columnas numéricas y categóricas.
    3) Para las numéricas: imputar valores faltantes con la mediana y luego escalar.
    4) Para las categóricas: aplicar one-hot encoding (variables 0/1).
    5) Unir todo en un solo DataFrame numérico, listo para modelos.
    """

    # 1) Eliminar columnas/filas vacías y duplicados
    df = df.dropna(axis=1, how='all')   # elimina columnas donde todos los valores son NaN
    df = df.dropna(axis=0, how='all')   # elimina filas donde todos los valores son NaN
    df = df.drop_duplicates()           # elimina filas repetidas

    # 2) Detectar columnas numéricas y categóricas
    cols_num = df.select_dtypes(include=[np.number]).columns
    cols_cat = df.select_dtypes(include=['object', 'string', 'category']).columns

    # 3) Imputar numéricas y escalarlas
    imputer_num = SimpleImputer(strategy='median')  # mediana es robusta a outliers
    scaler = StandardScaler()                       # deja media=0 y desviación estándar=1

    if len(cols_num) > 0:
        # 3.1) Imputar valores faltantes en columnas numéricas
        df_num = pd.DataFrame(
            imputer_num.fit_transform(df[cols_num]),
            columns=cols_num,
            index=df.index
        )

        # 3.2) Escalar columnas numéricas
        df_num_scaled = pd.DataFrame(
            scaler.fit_transform(df_num),
            columns=cols_num,
            index=df.index
        )
    else:
        df_num_scaled = pd.DataFrame(index=df.index)

    # 4) Codificar categóricas (one-hot encoding)
    # Cada categoría se convierte en una columna 0/1.
    if len(cols_cat) > 0:
        df_cat_encoded = pd.get_dummies(
            df[cols_cat],
            drop_first=False,
            dtype=int
        )
    else:
        df_cat_encoded = pd.DataFrame(index=df.index)

    # 5) Unir numéricas escaladas + categóricas codificadas
    df_final = pd.concat([df_num_scaled, df_cat_encoded], axis=1)

    return df_final


In [3]:
def detectar_problema(y):
    """
    Heurística para decidir si el problema es:
    - Clasificación: pocas clases (<= 15) o variable no numérica.
    - Regresión: variable numérica continua con muchas clases.
    """
    if pd.api.types.is_numeric_dtype(y):
        n_unicos = y.nunique()
        if n_unicos <= 15:
            return 'clasificacion'
        else:
            return 'regresion'
    else:
        return 'clasificacion'


def evaluar_modelos_clasificacion(X, y):
    """
    Clasificación:
    1) Define un diccionario de modelos.
    2) Para cada modelo, realiza validación cruzada (cv=5) con métrica accuracy.
    3) Devuelve un diccionario con el promedio de accuracy por modelo.
    """
    modelos = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForestClassifier': RandomForestClassifier(random_state=42),
        'SVC': SVC(probability=True)
    }
    resultados = {}

    for nombre, modelo in modelos.items():
        scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
        resultados[nombre] = scores.mean()

    return resultados, modelos


def evaluar_modelos_regresion(X, y):
    """
    Regresión:
        1) Define un diccionario de modelos.
        2) Para cada modelo, realiza validación cruzada (cv=5) con métrica R2.
        3) Devuelve un diccionario con el promedio de R2 por modelo.
    """
    modelos = {
        'LinearRegression': LinearRegression(),
        'RandomForestRegressor': RandomForestRegressor(random_state=42),
        'SVR': SVR()
    }
    resultados = {}

    for nombre, modelo in modelos.items():
        scores = cross_val_score(modelo, X, y, cv=5, scoring='r2')
        resultados[nombre] = scores.mean()

    return resultados, modelos


In [4]:
def graficos_exploratorios(df_limpio, df_raw):
    cols_num = df_limpio.select_dtypes(include=[np.number]).columns

    if len(cols_num) == 0:
        print('No hay columnas numéricas para graficar.')
        return

    # Histogramas
    print('Histogramas variables numéricas')
    df_limpio[cols_num].hist(figsize=(12, 8), bins=20)
    plt.tight_layout()
    plt.show()

    # Boxplots
    print('Boxplots variables numéricas')
    plt.figure(figsize=(12, 6))
    df_limpio[cols_num].plot(kind='box')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Pairplot (si pocas columnas)
    if len(cols_num) <= 6:
        print('Pairplot variables numéricas')
        sns.pairplot(df_limpio[cols_num])
        plt.show()

    # Frecuencias categóricas desde datos crudos
    cols_cat_raw = df_raw.select_dtypes(include=['object', 'string', 'category']).columns
    if len(cols_cat_raw) > 0:
        print('Frecuencias de una variable categórica (raw)')
        col_cat_sel = cols_cat_raw[0]
        print(f'Variable categórica usada para ejemplo: {col_cat_sel}')
        df_raw[col_cat_sel].value_counts().plot(kind='bar', rot=45, figsize=(6,4))
        plt.tight_layout()
        plt.show()


In [5]:
def graficos_modelo_clasificacion(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    print('Matriz de confusión (mejor modelo)')
    fig, ax = plt.subplots()
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues', ax=ax)
    plt.show()


def graficos_modelo_regresion(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)

    print('Predicho vs Real (mejor modelo)')
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_test, y_pred, alpha=0.7)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')
    ax.set_xlabel('Valores reales')
    ax.set_ylabel('Valores predichos')
    ax.set_title(f'R2 = {r2_score(y_test, y_pred):.3f}')
    plt.tight_layout()
    plt.show()


## Flujo completo: función `auto_analisis_csv`

Pasos dentro de la función:

1. Cargar el CSV original en un DataFrame (`df_raw`).
2. Mostrar primeras filas y estadísticas descriptivas.
3. Limpiar, codificar y normalizar (`limpiar_y_normalizar`) → `df_limpio`.
4. Generar gráficos exploratorios (histogramas, boxplots, pairplot, frecuencias categóricas).
5. Separar variables: `y` (target desde `df_raw`) y `X` (features desde `df_limpio`).
6. Detectar tipo de problema (clasificación vs regresión).
7. Dividir en train/test con `train_test_split`.
8. Evaluar varios modelos con validación cruzada (cv=5).
9. Elegir el mejor modelo según la métrica (accuracy o R2).
10. Entrenar el mejor modelo en train y generar gráficos de desempeño.
11. Guardar el CSV limpio/codificado/normalizado en disco.


In [6]:
def auto_analisis_csv(ruta_csv, nombre_target, ruta_salida='datos_limpios_codificados_normalizados.csv'):
    # 1) Cargar datos
    df_raw = pd.read_csv(ruta_csv)
    print('Vista previa de datos originales:')
    display(df_raw.head())

    print('Estadísticas descriptivas:')
    display(df_raw.describe(include='all'))

    # 2) Limpieza + codificación + normalización
    df_limpio = limpiar_y_normalizar(df_raw)
    print('Datos limpios/codificados/normalizados (primeras filas):')
    display(df_limpio.head())

    print('Columnas después de la codificación:')
    print(df_limpio.columns.tolist())

    # 3) Gráficos exploratorios
    graficos_exploratorios(df_limpio, df_raw)

    # 4) Separar X, y (target desde raw, features desde df_limpio)
    y = df_raw[nombre_target]
    X = df_limpio.copy()

    # 5) Detectar tipo problema
    tipo = detectar_problema(y)
    print(f'Tipo de problema detectado: {tipo}')

    # 6) Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 7) Evaluar modelos
    if tipo == 'clasificacion':
        resultados, modelos = evaluar_modelos_clasificacion(X_train, y_train)
        metrica = 'accuracy'
    else:
        resultados, modelos = evaluar_modelos_regresion(X_train, y_train)
        metrica = 'R2'

    # 8) Mostrar resultados
    print('Resultados de modelos:')
    for nombre, score in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
        print(f'{nombre}: {score:.4f} ({metrica})')

    # 9) Mejor modelo
    mejor_nombre = max(resultados, key=resultados.get)
    mejor_score = resultados[mejor_nombre]
    mejor_modelo = modelos[mejor_nombre]
    mejor_modelo.fit(X_train, y_train)

    print(f'Mejor modelo: {mejor_nombre} con {metrica} = {mejor_score:.4f}')
    print('- Accuracy (clasificación): proporción de aciertos.')
    print('- R2 (regresión): proporción de variabilidad explicada por el modelo.')

    # 10) Gráficos del mejor modelo
    if tipo == 'clasificacion':
        graficos_modelo_clasificacion(mejor_modelo, X_test, y_test)
    else:
        graficos_modelo_regresion(mejor_modelo, X_test, y_test)

    # 11) Guardar dataset limpio
    df_limpio.to_csv(ruta_salida, index=False)
    print(f'CSV limpio/codificado/normalizado guardado en: {ruta_salida}')

    return {
        'tipo_problema': tipo,
        'resultados_modelos': resultados,
        'mejor_modelo_nombre': mejor_nombre,
        'mejor_score': mejor_score,
        'metrica': metrica,
        'df_limpio': df_limpio,
        'modelo_entrenado': mejor_modelo
    }


In [7]:
# Ejemplo de uso:
# 1) Cambia la ruta y la columna target:
# ruta = 'ruta/a/tu_archivo.csv'          # por ejemplo: 'data/clientes.csv'
# target = 'nombre_de_la_columna_target'  # por ejemplo: 'compra', 'precio', etc.
#
# 2) Ejecuta el análisis automático:
# info = auto_analisis_csv(ruta, target)
#
# 3) (Opcional) ver el DataFrame limpio:
# info['df_limpio'].head()
